***Protein Protein Interaction Project***

*Authors*: Jordan Harrison, Haoyue Hu

In [26]:
# Import necessary Libraries

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef
import torch 

In [27]:
# Load the dataset
loaded = False

"""
have this in just as a way to check if the files are in the right place. the github is structured to have two data
folders (don't ask me why I don't know), but the when we submit the files could be in the same dir
"""
try:
    int_data = pd.read_csv("Data/Data/Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("Data/Data/SEQ-Supp-ABCD.tsv", sep='\t', header = None, names = ['id', 'seq'])
    seq_map = dict(zip(seq_data.id, seq_data.seq))
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Checking local directory for files...")
try:
    int_data = pd.read_csv("Supp-AB.tsv", sep='\t')
    seq_data = pd.read_csv("SEQ-Supp-ABCD.tsv", sep='\t', header = None)
    print("Data loaded successfully.")
except FileNotFoundError as e:
    print("Files not found in local directory")

# Check everything loaded correctly
pair_ids = set(int_data.v1) | set(int_data.v2)
print(f"Number of interacting protein pairs: {len(int_data)}")
print(f"Number of protein sequences: {len(seq_data)}")
print(f"Number of unique protein pairs: {len(pair_ids)}")

# Check data set
print(int_data.head())
#int_data.info()
print(seq_data.head())
#seq_data.info()

Data loaded successfully.
Files not found in local directory
Number of interacting protein pairs: 73108
Number of protein sequences: 10370
Number of unique protein pairs: 10364
               v1              v2  label
0     NP_663777.1     NP_001233.1      1
1     NP_055436.2     NP_001420.2      1
2     NP_003630.1  NP_001073594.1      1
3  NP_001001998.1     NP_067000.1      1
4     NP_001888.2     NP_001782.1      1
       id                                                seq
0  A0AVK6  MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...
1  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
2  A2A3N6  MASEVPYASGMPIKKIGHRSVDSSGGTTSSALKGAIQLGITHTVGS...
3  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...
4  A6H8Y1  MFRRARLSVKPNVRPGVGARGSTASNPQRGRESPRPPEPATDSASK...


In [28]:
# create new dataframe with the protein sequences and their interactions
df = int_data.copy()
df['protein_A_seq'] = df.v1.map(seq_map)
df['protein_B_seq'] = df.v2.map(seq_map)

"""
To avoid redunancy, the pair AB and BA are the same thing so we need to check for any duplicates
"""
# wizardry here, creates columns 'a' and 'b' then uses the min and max (which works alphabetically) to order the columns
# if you get ab and the ba this function puts ab into both of the new columns B)
df['a'] = df[['v1', 'v2']].min(axis=1)
df['b'] = df[['v1', 'v2']].max(axis=1)
df = df.drop_duplicates(subset=['a','b','label']).reset_index(drop=True)

#Drop weird amino acids
bad = df.protein_A_seq.str.contains('[UX]') | df.protein_B_seq.str.contains('[UX]')
df = df[~bad]

# clean it up, apply good labels, see the balance
df = df.rename(columns={'v1':'id_A','v2':'id_B'})[['id_A','id_B','protein_A_seq','protein_B_seq','label']]
print(df.label.value_counts(), df.shape)
df.head()



label
1    36620
0    36323
Name: count, dtype: int64 (72943, 5)


,id_A,id_B,protein_A_seq,protein_B_seq,label
0,NP_663777.1,NP_001233.1,MESSKKMDSPGALQTNPPLKLHTDRSAGTPVFVPEQGGYKEKFVKT...,MARPHPWWLCVLGTLVGLSATPAPKSCPERHYWAQGKLCCQMCEPG...,1
1,NP_055436.2,NP_001420.2,MVMSSYMVNSKYVDPKFPPCEEYLQGGYLGEQGADYYGGGAQGADF...,MAENVVEPGPPSAKRPKLSSPALSASASDGTDFGSLFDLEHDLPDE...,1
2,NP_003630.1,NP_001073594.1,MNRHLWKSQLCEMVQPSGGPAADQDVLGEESPLGKPAMLHLPSEQG...,MEGGRRARVVIESKRNFFLGAFPTPFPAEHVELGRLGDSETAMVPG...,1
3,NP_001001998.1,NP_067000.1,MAPPSTREPRVLSATSATKSDGEMVLPGFPDADSFVKFALGSVVAV...,MLFYSFFKSLVGKDVVVELKNDLSICGTLHSVDQYLNIKLTDISVT...,1
4,NP_001888.2,NP_001782.1,MQSGPRPPLPAPGLALALTLTMLARLASAASFFGENHLEVPVATAL...,MQTIKCVVVGDGAVGKTCLLISYTTNKFPSEYVPTVFDNYAVTVMI...,1


In [29]:
# Setup splits

"""
I want to try two things here:
1.) A random split, just pick random pairs for the train, validation, and test set
2.) Split based on proteins, ie randomly split the data set so it sees proteins that it has never seen in the test/validation sets
    this should in theory prevent the model from memorizing the "sequences"
"""
data = df.to_numpy()
X = data[:,2:4]
y = data[:,4]
labels = data[:,0:2]

# Option 1, random split

X_train, X_test, y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Option 2, split via proteins and then form sets

# Sort the proteins by labels
proteins = np.array(sorted(set(df.id_A) | set(df.id_B)))

#Initialize Group Shuffle Split
gss = GroupShuffleSplit(n_splits = 1, test_size = 0.3, random_state = 42)

#Use the set function to drop duplicates
gss_idx_train, gss_idx_test = next(gss.split(proteins, groups = proteins))
gss_train_prots, gss_test_prots = set(proteins[gss_idx_train]), set(proteins[gss_idx_test])

#check if the proteins are in the set
a_train, b_train = df.id_A.isin(gss_train_prots), df.id_B.isin(gss_train_prots)

#Keep unique entrys (both True, or both False) to form the training and test set
gss_X_train = df[a_train & b_train]
gss_X_test = df[~a_train & ~b_train]

print(len(gss_X_train), len(gss_X_test), (a_train ^ b_train).sum())


35582 6660 30701


In [30]:
# Featurize this sucker
"""
Turn each amino acid into a number, 1-20
Also consider calculating hydrophobicity, charge?
Break into polar, non-polar, acidic, basic?
Biopython is your friend
"""
# Define all the a.a
AA = "ACDEFGHIKLMNPQRSTVWY"

# makes a list of all possible dipeptides, ie AA, AC, AD, AE, ... YY
dipep = [''.join(p) for p in product('ACDEFGHIKLMNPQRSTVWY', repeat=2)]

def featurize_sequence(seq: str):
    """ 
    Featurize a protein sequnce into a numerical representation:

    This function contains the following (update as we go):
    1.) Amino acid composition (AAC): the percent of a.a in the seq
    2.) Dipeptide composition: the percent of each dipeptide in the seq
    3.) Secondary structure fraction
    4.) log of the length of the seq
    5.) log of the molecular weight
    6.) isoelectric point
    7.) Charge at pH 7.4
    8.) GRAVY (hydrophobicity)
    9.) Aromaticity
    10.) Instability index
    """
    pa = ProteinAnalysis(seq)
    aac = pa.amino_acids_percent
    dip = {d: 0 for d in dipep}
    for i in range(len(seq) - 1):
        dip[seq[i:i+2]] += 1
    dip_f = np.array([dip[d] for d in dipep]) / max(len(seq) - 1, 1)
    
    sec = pa.secondary_structure_fraction()
    
    scalers = [np.log(len(seq)), np.log(pa.molecular_weight()), pa.isoelectric_point(),
               pa.charge_at_pH(7.4), pa.gravy(), pa.aromaticity(),
               pa.instability_index()]
    return np.array([aac[aa] for aa in AA]) / 100, dip_f, np.array(scalers)

# Featurize the sequences
uniq = pd.concat([df[['id_A','protein_A_seq']].set_axis(['id','seq'], axis=1),
                  df[['id_B','protein_B_seq']].set_axis(['id','seq'], axis=1)]).drop_duplicates('id')
feats = {r.id: featurize_sequence(r.seq) for r in uniq.itertuples()}

print(len(feats))
# Print the first 2 entries
for i, (key, value) in enumerate(feats.items()):
  if i >= 2:
    break
  print(f"{key}: {value}")
    


10357
NP_663777.1: (array([0.04753521, 0.04929577, 0.0528169 , 0.07570423, 0.0334507 ,
       0.04401408, 0.0334507 , 0.03697183, 0.08978873, 0.09683099,
       0.02816901, 0.03697183, 0.04049296, 0.0528169 , 0.04929577,
       0.0915493 , 0.04049296, 0.06866197, 0.00704225, 0.02464789]), array([0.00176367, 0.00352734, 0.00352734, 0.00352734, 0.00176367,
       0.00352734, 0.00352734, 0.        , 0.        , 0.00881834,
       0.        , 0.        , 0.00176367, 0.00176367, 0.00352734,
       0.00529101, 0.00176367, 0.00352734, 0.        , 0.        ,
       0.00352734, 0.00176367, 0.        , 0.00352734, 0.        ,
       0.00176367, 0.00352734, 0.        , 0.00705467, 0.        ,
       0.00176367, 0.        , 0.00529101, 0.00176367, 0.00176367,
       0.00881834, 0.00176367, 0.00705467, 0.        , 0.        ,
       0.00352734, 0.00529101, 0.00176367, 0.        , 0.        ,
       0.00176367, 0.00176367, 0.00176367, 0.00705467, 0.00529101,
       0.00176367, 0.00176367, 0.0035273

In [31]:
# Train the model

scaler_it = StandardScaler().fit(np.stack([feats[p][2] for p in gss_train_prots]))

prot_vec = {p: np.concatenate([aac, dip, sc]) for p, (aac, dip, sc) in feats.items()}
known = set(prot_vec)

# 
def pair_features(d, mode="pair"):
    d = d[d.id_A.isin(known) & d.id_B.isin(known)]   # skip proteins that failed featurization
    fA = np.stack(d.id_A.map(prot_vec).values).astype(np.float32)
    fB = np.stack(d.id_B.map(prot_vec).values).astype(np.float32)
    if mode == "pair":        # symmetric: swapping A and B gives the same features
        X = np.hstack([fA + fB, np.abs(fA - fB)])
    else:                     # "A_only", for the shortcut check
        X = fA
    return X, d.label.astype(int).values

def evaluate(model, X, y, name):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print(f"{name:14s} acc={accuracy_score(y, pred):.3f}  AUROC={roc_auc_score(y, prob):.3f}  "
          f"AUPRC={average_precision_score(y, prob):.3f}  MCC={matthews_corrcoef(y, pred):.3f}")

Xtr, ytr = pair_features(gss_X_train)
Xte, yte = pair_features(gss_X_test)
print(Xtr.shape, Xte.shape)

rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rf.fit(Xtr, ytr)

evaluate(rf, Xtr, ytr, "train")
evaluate(rf, Xte, yte, "test (disjoint)")

(35582, 854) (6660, 854)
train          acc=1.000  AUROC=1.000  AUPRC=1.000  MCC=1.000
test (disjoint) acc=0.689  AUROC=0.764  AUPRC=0.769  MCC=0.378
